In [ ]:
#SETTING
from google.colab import userdata, drive
import os

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Setup Info
GIT_TOKEN = userdata.get("My_Token")
GIT_USERNAME = "Tanior14"
GIT_EMAIL = "thuantan1905@gmail.com"
REPO_OWNER = "nhnminh1409"
GIT_REPO = "instacart-market-basket-analysis"

# Define Source File and Destination Path
SOURCE_PATH = "/content/drive/MyDrive/Colab Notebooks/ 04_feature_engineering.ipynb"
TARGET_DIR = "notebooks"

# 3.Setup Git Repo
%cd /content/
# Fix URL formatting (Remove redundant ://)
PUSH_URL = f"https://{GIT_TOKEN}@github.com/{REPO_OWNER}/{GIT_REPO}.git"

if not os.path.exists(GIT_REPO):
    !git clone {PUSH_URL}
else:
    print("Repo existed!")

%cd {GIT_REPO}
!git config --global user.email "{GIT_EMAIL}"
!git config --global user.name "{GIT_USERNAME}"

!git fetch origin
!git reset --hard origin/main

# 4. Copy files to the specific directory
# Create 'notebooks' directory if not exists
!mkdir -p {TARGET_DIR}

# Move files to 'notebooks' directory
!cp "{SOURCE_PATH}" "{TARGET_DIR}/"

# 5. Deploy code to GitHub repository
!git add .
!git commit -m "feat: complete user behavior EDA and feature engineering" || echo "No changes to commit"
!git push origin main --force

In [2]:
print('Building USER–PRODUCT FEATURES...')

# ── 1. Core user–product interaction stats ───────────────────────────────────
upf = df.groupby(['user_id', 'product_id']).agg(
    up_total_purchases      = ('order_id',          'count'),
    up_reorder_count        = ('reordered',          'sum'),
    up_avg_add_to_cart      = ('add_to_cart_order',  'mean'),
    up_min_add_to_cart      = ('add_to_cart_order',  'min'),
).reset_index()

# ── 2. Reorder rate per user–product pair ────────────────────────────────────
upf['up_reorder_rate'] = (
    upf['up_reorder_count'] / upf['up_total_purchases']
).astype(np.float32)

# ── 3. Order number of last purchase (recency signal) ────────────────────────
last_order = (
    df.groupby(['user_id', 'product_id'])['order_number']
    .max()
    .reset_index()
    .rename(columns={'order_number': 'up_last_order_number'})
)

# Total orders the user has made
user_max_order = (
    orders_dedup.groupby('user_id')['order_number']
    .max()
    .reset_index()
    .rename(columns={'order_number': 'user_max_order_number'})
)

upf = upf.merge(last_order,      on=['user_id', 'product_id'], how='left')
upf = upf.merge(user_max_order,  on='user_id',                 how='left')

# Orders since last purchase (= 0 means bought in most recent order)
upf['up_orders_since_last_purchase'] = (
    upf['user_max_order_number'] - upf['up_last_order_number']
).astype(np.int16)

upf.drop(columns=['up_last_order_number', 'user_max_order_number'], inplace=True)

# ── 4. Purchase rate = up_total_purchases / user_total_orders ─────────────────
user_order_counts = uf[['user_id', 'user_total_orders']]
upf = upf.merge(user_order_counts, on='user_id', how='left')

upf['up_order_rate'] = (
    upf['up_total_purchases'] / upf['user_total_orders']
).astype(np.float32)

upf.drop(columns=['user_total_orders'], inplace=True)

# ── 5. Memory optimization ───────────────────────────────────────────────────
upf = reduce_mem_usage(upf)

print(f'\n✅ user_product_features shape: {upf.shape}')
upf.head()

Building USER–PRODUCT FEATURES...


NameError: name 'df' is not defined